---
---
# 🏗️ Self-Assignment — Mini RAG Project

> **Complete this after class.** Build a small but complete RAG system on a document of your choice, following the same architecture demonstrated in the session.

---

## 🎯 Project Overview

You will build a **Document Q&A Assistant** that:
1. Loads and chunks **your own document** (PDF, TXT, or plain text)
2. Indexes it into ChromaDB using Azure embeddings
3. Answers questions using a **grounded RAG pipeline**
4. Supports **hybrid search** (BM25 + dense)
5. Returns **cited, structured answers**
6. Handles failure modes gracefully

---

## 📐 Architecture You Are Building

```
Your Document (PDF/TXT)
        │
   [Section 1] Load & chunk
        │
   [Section 2] Embed → ChromaDB
        │              │
   User Query ──► [Section 3] Hybrid Search (BM25 + Dense + RRF)
                       │
                  [Section 4] Build grounded prompt
                       │
                  Azure OpenAI (gpt-4o-mini)
                       │
                  [Section 5] Structured answer + citations
                       │
                  [Section 6] Evaluation & reflection
```

---

## 📋 Grading Rubric

| Section | Task | Points |
|---------|------|--------|
| 1 | Document loaded, chunked correctly, chunk stats printed | 15 |
| 2 | Vectors stored in ChromaDB, metadata tagged | 15 |
| 3 | Hybrid search working, BM25 + Dense + RRF fused | 20 |
| 4 | Grounded system prompt, 5 questions answered | 20 |
| 5 | Citation JSON returned and parsed correctly | 15 |
| 6 | At least 2 failure modes demonstrated with fixes | 15 |
| **Total** | | **100** |

---

## 🔖 How to Submit
1. Complete all `# TODO` cells below
2. Run the entire notebook from top to bottom (Runtime → Run All)
3. Save as: `RAG_Assignment_<YourName>.ipynb`
4. Upload to the class portal

> ⚠️ **Important:** All outputs must be visible when you submit. Do NOT clear outputs before submitting.

---
## Step 0 — Choose Your Document

Pick **one** of the following options for your source document.
Choose something you find interesting — you will be asking 5 questions about it!

| Option | Type | Example |
|--------|------|---------|
| A | Company annual/quarterly report | Apple Q4 2024, any public company 10-K |
| B | Research paper or article | Any PDF from arXiv, WHO, World Bank |
| C | Product documentation | API docs, user manual, technical spec |
| D | News article collection | 3–5 articles on the same topic as one text |
| E | Wikipedia article (long-form) | Copy a long Wikipedia article as plain text |

🔑 **Minimum length:** at least 800 words (~5,000 characters) so you get enough chunks to make retrieval meaningful.

In [1]:
# Install dependencies required for our stand-alone clean running
!pip install -q chromadb rank_bm25 google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [23]:
MY_DOCUMENT = """
LAPORAN RILIS SURVEI NASIONAL INDIKATOR POLITIK INDONESIA
EVALUASI PUBLIK ATAS KINERJA SATU TAHUN PEMERINTAHAN PRABOWO-GIBRAN
Periode Survei Tatap Muka: 20-27 Oktober 2025

METODOLOGI SURVEI
Populasi survei ini adalah seluruh warga negara Indonesia yang punya hak pilih dalam pemilihan umum, yakni mereka yang sudah berumur 17 tahun atau lebih, atau sudah menikah ketika survei dilakukan. Penarikan sampel menggunakan metode multistage random sampling. Dalam survei ini jumlah sampel sebanyak 1220 orang. Dengan asumsi metode simple random sampling, ukuran sampel 1.220 responden memiliki toleransi kesalahan (margin of error--MoE) sekitar kurang lebih 2.9% pada tingkat kepercayaan 95 persen. Sampel berasal dari seluruh Provinsi di Indonesia yang terdistribusi secara proporsional. Responden terpilih diwawancarai lewat tatap muka oleh pewawancara yang telah dilatih. Quality control terhadap hasil wawancara dilakukan secara random sebesar 20% dari total sampel oleh supervisor dengan kembali mendatangi responden terpilih (spot check). Dalam quality control tidak ditemukan kesalahan berarti.

PROFIL DEMOGRAFI RESPONDEN
Profil gender sampel adalah Laki-Laki 50.0% dan Perempuan 50.0%. Berdasarkan wilayah tinggal, responden berada di Pedesaan sebesar 50.3% dan Perkotaan sebesar 49.7%. Profil usia responden adalah: di bawah 20 tahun 9.2%, 21-25 tahun 10.9%, 26-30 tahun 10.8%, 31-35 tahun 10.6%, 36-40 tahun 10.8%, 41-45 tahun 10.0%, 46-50 tahun 9.3%, 51-55 tahun 8.2%, 56-60 tahun 6.8%, dan di atas 60 tahun 13.5%. Profil pendidikan responden adalah: lulusan SD ke bawah 37.0%, SLTP 18.0%, SLTA 31.2%, dan Kuliah/Perguruan Tinggi 13.8%. Profil agama responden adalah: Islam 87.2%, Protestan/Katolik 10.0%, dan Lainnya 2.8%. Distribusi etnis responden meliputi: Jawa 40.8%, Sunda 15.5%, Batak 3.7%, Madura 3.1%, Betawi 2.9%, Minang 2.8%, Bugis 2.6%, Melayu 2.3%, dan Etnis Lainnya 26.4%.

EVALUASI KONDISI UMUM NASIONAL
Kondisi Ekonomi Nasional: Kondisi ekonomi lebih banyak dinilai sedang yaitu 42.4%, sementara yang menilai baik atau sangat baik adalah 31.1% (Sangat Baik 3.5%, Baik 27.6%), dan yang menilai buruk atau sangat buruk adalah 26.2% (Buruk 22.6%, Sangat Buruk 3.6%). Sekitar 0.3% responden tidak menjawab.
Kondisi Politik Nasional: Kondisi politik lebih banyak dinilai sedang yaitu 36.8%, sementara yang menilai baik atau sangat baik adalah 31.0% (Sangat Baik 2.4%, Baik 28.6%), dan yang menilai buruk atau sangat buruk adalah 24.6% (Buruk 21.8%, Sangat Buruk 2.8%). Sekitar 7.5% responden tidak menjawab.
Kondisi Keamanan Nasional: Kondisi keamanan lebih banyak dinilai baik atau sangat baik yaitu 56.5% (Sangat Baik 5.2%, Baik 51.3%), kemudian yang menilai sedang yaitu 27.4%, dan yang menilai buruk atau sangat buruk adalah 15.0% (Buruk 13.5%, Sangat Buruk 1.5%). Sekitar 1.0% responden tidak menjawab.
Kondisi Penegakan Hukum: Kondisi penegakan hukum lebih banyak dinilai baik atau sangat baik yaitu 40.8% (Sangat Baik 3.0%, Baik 37.8%), sementara yang menilai sedang ada 29.3%, dan yang menilai buruk atau sangat buruk ada 26.4% (Buruk 23.1%, Sangat Buruk 3.3%). Sekitar 3.4% tidak menjawab.
Kondisi Pemberantasan Korupsi: Pemberantasan korupsi lebih banyak dinilai baik atau sangat baik yaitu 42.7% (Sangat Baik 7.2%, Baik 35.5%), kemudian yang menilai buruk atau sangat buruk ada 30.0% (Buruk 25.1%, Sangat Buruk 4.9%), dan yang menilai sedang 22.5%. Sekitar 4.8% tidak menjawab.
Kondisi Ekonomi Rumah Tangga Dibanding Tahun Lalu: Kondisi ekonomi rumah tangga dibanding tahun lalu lebih banyak dinilai tidak ada perubahan sebesar 36.5%. Yang menilai membaik atau membaik secara signifikan adalah 38.3% (Lebih Baik 30.3%, Jauh Lebih Baik 8.0%), lebih banyak dibanding yang menilai memburuk sebesar 25.2% (Jauh Lebih Buruk 4.0%, Lebih Buruk 21.2%). Sekitar 0.1% tidak menjawab.

MASALAH PALING MENDESAK BAGI BANGSA
Menurut responden, empat masalah paling mendesak yang harus diselesaikan oleh pemimpin nasional lima tahun ke depan adalah:
1. Pemberantasan korupsi sebesar 25.5%
2. Mengendalikan harga-harga kebutuhan pokok sebesar 22.3%
3. Menyediakan lapangan kerja atau mengatasi pengangguran sebesar 15.6%
4. Mengurangi kemiskinan sebesar 13.4%
Keempat masalah tersebut merupakan masalah paling mendesak menurut total 76.8% warga. Masalah lainnya meliputi: Keamanan/ketertiban 4.5%, Memajukan sektor pertanian 3.3%, Pembangunan/perbaikan infrastruktur jalan dan jembatan 2.4%, Pemerataan pendapatan 1.9%, Meningkatkan kualitas pendidikan 1.8%, Mendorong pertumbuhan UMKM 1.4%, Kemudahan mendapatkan modal usaha 0.9%, Hutang luar negeri 0.8%, Mencegah masuknya barang dan pekerja asing 0.5%, Memperkuat nilai tukar rupiah 0.4%, Politik dinasti 0.3%, Perumahan rakyat 0.3%, Meningkatkan kualitas layanan kesehatan 0.3%.

KEPUASAN ATAS KINERJA PRESIDEN PRABOWO SUBIANTO
Secara umum, tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto dalam satu tahun masa jabatannya sangat tinggi. Sekitar 77.7% warga merasa puas (Sangat Puas 17.3%, Cukup Puas 60.4%). Sementara itu, ada sekitar 20.8% warga yang merasa kurang puas atau tidak puas sama sekali (Kurang Puas 19.8%, Tidak Puas Sama Sekali 1.0%). Sekitar 1.5% responden tidak memberikan jawaban.

ANALISIS KEPUASAN BERDASARKAN DEMOGRAFI DAN WILAYAH
Berdasarkan gender, kepuasan pada laki-laki mencapai 81.4%, sedangkan perempuan mencapai 73.9%. Berdasarkan generasi, kepuasan tertinggi di kalangan Gen Z (1997-2012) sebesar 81.8%, diikuti Millenials (1981-1996) 77.1%, Gen X (1965-1980) 75.8%, dan Baby Boomers (1918-1964) sebesar 74.5%. Berdasarkan wilayah geografi, tingkat kepuasan di Sulawesi tercatat sebesar 91.7%, Kalimantan sebesar 91.0%, Bali dan Nusa Tenggara sebesar 89.4%, Jawa Timur sebesar 86.0%, Banten sebesar 84.0%, Jawa Barat sebesar 78.4%, Jawa Tengah dan DIY sebesar 71.5%, Sumatera sebesar 68.0%, DKI Jakarta sebesar 62.8%, dan wilayah Maluku serta Papua tercatat paling rendah yaitu sebesar 59.8%.

ALASAN UTAMA KEPUASAN KINERJA PRESIDEN
Di antara responden yang merasa puas dengan kinerja Presiden Prabowo Subianto, alasan utama yang paling banyak dikemukakan adalah:
1. Upaya memberantas korupsi yang dinilai tegas sebesar 19.5%
2. Kinerjanya yang dinilai sudah bagus dan ada bukti nyata sebesar 15.9%
3. Karakter yang tegas, berwibawa, dan berani sebesar 12.8%
4. Sering memberikan bantuan sosial kepada masyarakat sebesar 9.7%
5. Adanya program unggulan Makan Bergizi Gratis (MBG) sebesar 8.0%
6. Program kerja yang dinilai secara umum bagus sebesar 6.1%

EVALUASI PROGRAM PEMBANGUNAN RUMAH SAKIT DI DAERAH
Program pembangunan Rumah Sakit (RS) lengkap berkualitas di daerah diketahui oleh 19.7% warga, sedangkan mayoritas 80.3% belum tahu. Di antara mereka yang mengetahui program ini, tingkat kepuasan sangat tinggi yaitu 82.5% menyatakan sangat puas atau cukup puas, sementara hanya 12.6% yang kurang atau tidak puas sama sekali, dan 4.8% tidak menjawab.

EVALUASI PROGRAM PENUNTASAN PENYAKIT TBC
Program Penuntasan Penyakit TBC diketahui oleh sekitar 29.0% warga, sedangkan 71.0% lainnya belum mengetahui. Di antara kelompok warga yang mengetahui program ini, mayoritas merasa puas dengan rincian: Sangat Puas 14.9%, Cukup Puas 67.3% (total kepuasan 82.2%), Kurang Puas 11.6%, Tidak Puas Sama Sekali 1.0%, dan 5.2% tidak memberikan jawaban.

EVALUASI PROGRAM PENDIDIKAN DAN RENOVASI SEKOLAH
Program Renovasi Sekolah diketahui oleh sekitar 38.5% warga, sedangkan 61.5% lainnya belum tahu. Di antara warga yang mengetahui program ini, mayoritas merasa puas dengan rincian: Sangat Puas 11.8%, Cukup Puas 67.9% (total kepuasan 79.7%), Kurang Puas 15.1%, Tidak Puas Sama Sekali 1.2%, dan 3.9% tidak memberikan jawaban.
Program Sekolah Rakyat diketahui oleh sekitar 32.5% warga, sedangkan 67.5% lainnya tidak tahu. Kepuasan di antara yang tahu adalah: Sangat Puas 17.0%, Cukup Puas 58.5% (total 75.5%), Kurang Puas 13.0%, Tidak Puas Sama Sekali 1.8%, dan tidak tahu/tidak jawab 9.7%.
Program Sekolah Garuda diketahui oleh hanya sekitar 8.6% warga, sedangkan mayoritas 91.4% tidak tahu. Kepuasan di antara yang tahu adalah: Sangat Puas 10.9%, Cukup Puas 59.6% (total 70.5%), Kurang Puas 8.8%, Tidak Puas Sama Sekali 2.9%, dan tidak tahu/tidak jawab 17.8%.

EVALUASI KOPERASI DESA (KOPDES) MERAH PUTIH
Program pembentukan Koperasi Desa (Kopdes) Merah Putih diketahui oleh sekitar 46.2% warga, sementara 53.8% lainnya tidak mengetahui. Di antara warga yang mengetahui program tersebut, tingkat kepuasan mencapai 48.5% (Sangat Puas 5.7%, Cukup Puas 42.7%), sedangkan yang kurang puas mencapai 24.5%, tidak puas sama sekali 4.5%, dan sekitar 22.5% menyatakan tidak tahu atau tidak menjawab.

EVALUASI PROGRAM TIGA JUTA RUMAH
Program Tiga Juta Rumah diketahui oleh sekitar 19.2% warga, sedangkan mayoritas 80.8% tidak tahu. Di antara yang tahu, tingkat kepuasan tercatat sebesar 51.8% (Sangat Puas 5.5%, Cukup Puas 46.3%), sedangkan yang merasa kurang puas sebesar 26.6%, tidak puas sama sekali 5.8%, dan tidak tahu/tidak jawab sebesar 15.8%.

EVALUASI PROGRAM LUMBUNG PANGAN NASIONAL, DAERAH DAN DESA
Program Lumbung Pangan Nasional, Daerah dan Desa diketahui oleh sekitar 20.0% warga, sedangkan 80.0% tidak tahu. Di antara warga yang mengetahui, tingkat kepuasan mencapai 67.3% (Sangat Puas 6.6%, Cukup Puas 60.7%), sedangkan yang kurang puas 20.4%, tidak puas sama sekali 1.9%, dan tidak tahu/tidak jawab sebesar 10.4%.

AWARENESS DAN SIKAP PUBLIK TERHADAP KEBIJAKAN EFISIENSI ANGGARAN
Kebijakan pemangkasan belanja negara atau efisiensi anggaran belanja yang dilakukan oleh Presiden Prabowo Subianto diketahui oleh sekitar 31.7% warga, sementara 68.3% lainnya tidak mengetahui. Di antara warga yang mengetahui, mayoritas sangat menyetujui kebijakan ini dengan rincian: Sangat Setuju 4.0%, Setuju 57.1% (total setuju 61.1%), Kurang Setuju 16.7%, Tidak Setuju Sama Sekali 3.8%, dan tidak tahu/tidak jawab 18.4%.
Terkait pendapat mengenai kebijakan efisiensi anggaran ini:
1. Sekitar 56.7% berpendapat efisiensi anggaran Pemerintahan Prabowo sudah tepat karena memangkas banyak pemborosan APBN dan anggaran yang tidak menyentuh kebutuhan masyarakat secara langsung (seperti perjalanan dinas, ATK, dll).
2. Sekitar 29.1% berpendapat efisiensi anggaran Pemerintahan Prabowo kurang tepat dijalankan saat ini karena di tengah melemahnya daya beli masyarakat, belanja pemerintah menjadi sangat penting untuk menjaga pertumbuhan ekonomi di tengah situasi sulit.
3. Sekitar 14.3% tidak tahu atau tidak menjawab.
Terkait efektivitas program efisiensi belanja negara ini, sebanyak 61.7% berpendapat kebijakan ini telah mengurangi pemborosan APBN dan mengalokasikan lebih banyak anggaran untuk kepentingan masyarakat secara langsung. Sementara 21.3% menilai kebijakan ini lebih banyak digunakan untuk kepentingan politik dan janji kampanye yang kurang produktif untuk perekonomian rakyat, dan 17.0% tidak tahu/tidak menjawab.

AWARENESS DAN SIKAP PUBLIK TERHADAP PROGRAM MAGANG NASIONAL
Program Magang Nasional selama 6 bulan bagi lulusan perguruan tinggi (Diploma dan Sarjana) untuk mendapatkan pengalaman kerja dengan gaji setara UMK (Upah Minimum Kota/Kabupaten) setiap bulannya diketahui oleh sekitar 20.8% warga, sementara 79.2% tidak tahu. Sikap publik terhadap program ini sangat positif: dari semua responden, 6.9% sangat setuju dan 65.5% setuju (total setuju 72.4%). Di antara mereka yang mengetahui program ini, tingkat persetujuan bahkan mencapai 80.3%.

AWARENESS DAN SIKAP PUBLIK TERHADAP PROGRAM BANTUAN SOSIAL (BANSOS) PANGAN
Program bantuan sosial (bansos) pangan berupa pemberian bantuan beras 20 Kg dan minyak goreng 4 liter selama dua bulan (Oktober dan November) bagi masyarakat berpenghasilan rendah diketahui oleh sekitar 50.5% warga, sementara 49.5% lainnya tidak tahu. Sikap publik terhadap program ini sangat mendukung: secara umum, 16.0% sangat setuju dan 70.8% setuju (total setuju 86.8%). Di antara mereka yang mengetahui program ini, tingkat persetujuan mencapai 74.3%.

AWARENESS DAN SIKAP PUBLIK TERHADAP PROGRAM BANTUAN LANGSUNG TUNAI (BLT)
Kebijakan pemberian tambahan Bantuan Langsung Tunai (BLT) senilai Rp. 300.000,- per bulan kepada lebih dari 35 juta keluarga penerima manfaat sejak bulan Oktober hingga Desember 2025 diketahui oleh sekitar 62.9% warga, sementara 37.1% lainnya tidak tahu. Sikap publik terhadap program ini sangat menyetujui: dari semua responden, 14.9% sangat setuju dan 71.9% setuju (total setuju 86.8%). Di antara kelompok yang mengetahui program ini, tingkat persetujuan mencapai 74.8%.

PENERIMAAN BANTUAN SOSIAL (BANSOS) SELAMA TAHUN 2025
Selama tahun 2025, sekitar 33.4% warga menyatakan bahwa mereka atau anggota keluarga di rumah pernah menerima bantuan sosial (bansos), sementara mayoritas 66.6% menyatakan tidak pernah menerima. Kepuasan kinerja presiden tercatat lebih tinggi secara signifikan di kelompok yang menerima bansos (80.5%) dibandingkan kelompok yang tidak pernah menerima bansos (76.2%). Di antara jenis bantuan sosial yang pernah diterima oleh penerima bansos selama 2025 adalah:
1. Bansos beras sebesar 58.0%
2. Program Keluarga Harapan (PKH) sebesar 44.4%
3. Bantuan Pangan Non Tunai (BPNT) sebesar 32.3%
4. Program Indonesia Pintar (PIP) sebesar 16.9%
5. Bantuan Subsidi Upah (BSU) sebesar 9.2%
Sekitar 1.5% menjawab tidak tahu atau tidak menjawab.

KESIMPULAN UMUM SURVEI
1. Terjadi perubahan yang signifikan terkait persepsi terhadap kondisi umum nasional, baik dari sisi ekonomi, politik, keamanan, dan penegakan hukum. Kemungkinan besar perubahan persepsi positif ini didorong oleh peluncuran kebijakan-kebijakan yang secara langsung menyentuh rakyat, terutama paket bantuan sosial (bansos) yang semakin masif didistribusikan.
2. Kondisi perekonomian dinilai mengalami perbaikan, saat ini lebih banyak masyarakat yang menilai positif ketimbang menilai negatif. Aspek perekonomian sangat fundamental karena paling erat kaitannya dengan kehidupan masyarakat sehari-hari. Ketika beban ekonomi masyarakat berkurang, secara psikologis menjadi semakin kondusif dalam memberikan persepsi positif terhadap pemerintah.
3. Penilaian terhadap kondisi politik, keamanan, pemberantasan korupsi, dan penegakan hukum juga mengalami perbaikan yang positif dan signifikan. Begitu pula evaluasi warga terhadap kinerja demokrasi nasional.
4. Program prioritas pemerintahan Prabowo-Gibran yang semula dinilai sangat kritis oleh publik kini tingkat kepuasannya sedikit membaik, sehingga dukungan warga atas keberlanjutan program saat ini juga tampak menguat. Kebijakan baru yang diluncurkan belakangan ini cepat merambat ke telinga warga dengan dukungan yang sangat tinggi.
5. Program terbesar pemerintah yaitu Makan Bergizi Gratis (MBG) sempat mendapat penolakan yang masif karena menyebabkan kapasitas fiskal pemerintah, terutama pemerintah daerah, menjadi lemah sehingga efek ungkit terhadap perekonomian secara luas dinilai rendah. Meski program MBG saat ini mengalami peningkatan persepsi positif, ke depan program ini sangat berpotensi menjumpai guncangan serupa karena kritik dasarnya sudah sangat besar. Karena itu, evaluasi berkala harus terus dilakukan guna memastikan efektivitasnya dalam mendorong penguatan perekonomian rakyat secara riil.
"""

DOC_NAME    = "survei_indikator_oktober_2025.txt"
DOC_TOPIC   = "Survei Nasional Evaluasi Satu Tahun Pemerintahan Prabowo-Gibran"

# Validation
assert len(MY_DOCUMENT.strip()) >= 500, \
    f"Document too short ({len(MY_DOCUMENT)} chars). Minimum 800 words required."

print(f"Document loaded: '{DOC_TOPIC}'")
print(f"   Name     : {DOC_NAME}")
print(f"   Size     : {len(MY_DOCUMENT):,} characters, ~{len(MY_DOCUMENT.split())} words")

Document loaded: 'Survei Nasional Evaluasi Satu Tahun Pemerintahan Prabowo-Gibran'
   Name     : survei_indikator_oktober_2025.txt
   Size     : 15,116 characters, ~2076 words


---
## Step 1 — Load & Chunk Your Document

Use the `recursive_split()` function from the demo session.

**Your task:**
- Choose `chunk_size` and `chunk_overlap` appropriate for your document type
- Print chunk statistics (count, average size, min, max)
- Display the first 3 chunks to verify the split looks sensible

**Guidance:**
- Dense prose (reports, articles): `chunk_size=400–512`
- Technical docs with lists: `chunk_size=250–350`
- Always use `chunk_overlap` ≈ 10–15% of `chunk_size`

In [24]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> list[str]:
    separators = ["\n\n", "\n", " ", ""]
    def split_recurse(txt, size, overlap, seps):
        if len(txt) <= size:
            return [txt]
        sep = seps[0]
        parts = txt.split(sep) if len(seps) > 1 else list(txt)

        chunks = []
        current = ""
        for part in parts:
            candidate = current + (sep if current else "") + part
            if len(candidate) <= size:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                if len(part) > size:
                    chunks.extend(split_recurse(part, size, overlap, seps[1:]))
                    current = ""
                else:
                    if overlap > 0 and current:
                        overlap_txt = current[-overlap:]
                        current = overlap_txt + sep + part
                    else:
                        current = part
        if current:
            chunks.append(current)
        return chunks
    return split_recurse(text, chunk_size, chunk_overlap, separators)


CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

assert CHUNK_SIZE > 0,    "Set CHUNK_SIZE (e.g. 400)"
assert CHUNK_OVERLAP > 0, "Set CHUNK_OVERLAP (e.g. 60)"


my_chunks = recursive_split(MY_DOCUMENT, CHUNK_SIZE, CHUNK_OVERLAP)

assert my_chunks is not None and len(my_chunks) >= 3, \
    "my_chunks must contain at least 3 chunks. Check your recursive_split() call."


sizes = [len(c) for c in my_chunks]
print(f"Created {len(my_chunks)} chunks")
print(f"   Avg size : {sum(sizes)/len(sizes):.1f} chars")
print(f"   Min size : {min(sizes)} chars")
print(f"   Max size : {max(sizes)} chars")


print("\nFirst 3 chunks:")
for i, chunk in enumerate(my_chunks[:3]):
    print(f"--- Chunk {i+1} ---")
    print(chunk)
    print()

Created 49 chunks
   Avg size : 335.3 chars
   Min size : 17 chars
   Max size : 528 chars

First 3 chunks:
--- Chunk 1 ---

LAPORAN RILIS SURVEI NASIONAL INDIKATOR POLITIK INDONESIA
EVALUASI PUBLIK ATAS KINERJA SATU TAHUN PEMERINTAHAN PRABOWO-GIBRAN
Periode Survei Tatap Muka: 20-27 Oktober 2025

--- Chunk 2 ---
METODOLOGI SURVEI

--- Chunk 3 ---
Populasi survei ini adalah seluruh warga negara Indonesia yang punya hak pilih dalam pemilihan umum, yakni mereka yang sudah berumur 17 tahun atau lebih, atau sudah menikah ketika survei dilakukan. Penarikan sampel menggunakan metode multistage random sampling. Dalam survei ini jumlah sampel sebanyak 1220 orang. Dengan asumsi metode simple random sampling, ukuran sampel 1.220 responden memiliki toleransi kesalahan (margin of error--MoE) sekitar kurang lebih 2.9% pada tingkat kepercayaan 95



---
## Step 2 — Embed & Index into ChromaDB

**Your task:**
- Embed all chunks using the `embed()` helper (Azure API)
- Create a new ChromaDB collection for your document
- Store chunks with meaningful **metadata** tags (at minimum: `source` and `section` or `chunk_index`)
- Run a test search to verify retrieval works

**Tip:** You can reuse `label_section()` from the demo, or write your own labelling logic for your document.

In [ ]:
# ── Step 2a: Embed all chunks via Google Gemini API ───────────────────────
import os
import google.generativeai as genai

# Attempt to configure Gemini from environment or Colab secrets
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = input("Enter your Google Gemini API Key: ")

genai.configure(api_key=GEMINI_API_KEY)

def embed(texts: list[str]) -> list[list[float]]:
    if isinstance(texts, str):
        texts = [texts]
    response = genai.embed_content(
        model="models/gemini-embedding-2",
        content=texts,
        task_type="retrieval_document"
    )
    return response['embedding']

print(f"Embedding {len(my_chunks)} chunks...")

# ── TODO: Call embed() and store the result ───────────────────────────────
my_vectors = embed(my_chunks)

assert my_vectors is not None, "Call embed(my_chunks) and assign the result to my_vectors"
assert len(my_vectors) == len(my_chunks), "Number of vectors must match number of chunks"

print(f"Embedded {len(my_vectors)} chunks")
print(f"   Vector dimensions : {len(my_vectors[0])}")
print(f"   Preview (chunk 0) : {[round(v, 4) for v in my_vectors[0][:5]]}...")

In [32]:
# ── Step 2b: Store in ChromaDB ────────────────────────────────────────────
import chromadb, shutil
import uuid
from pathlib import Path

MY_CHROMA_PATH = f"/tmp/my_rag_chroma_{uuid.uuid4().hex[:8]}"

my_chroma_client = chromadb.PersistentClient(path=MY_CHROMA_PATH)

class MyEmbeddingFn(chromadb.EmbeddingFunction):
    def __call__(self, input):
        return embed(input)

my_collection = my_chroma_client.create_collection(
    name="my_document",
    embedding_function=MyEmbeddingFn(),
    metadata={"hnsw:space": "cosine"},
)

# ── TODO: Write a function or logic to assign metadata to each chunk ───────
def my_metadata(chunk: str, index: int) -> dict:
    section = "Umum"
    if "METODOLOGI" in chunk:
        section = "Metodologi"
    elif "PROFIL DEMOGRAFI" in chunk:
        section = "Demografi"
    elif "EVALUASI KONDISI" in chunk:
        section = "Evaluasi Kondisi Nasional"
    elif "MENDESAK" in chunk:
        section = "Masalah Mendesak"
    elif "KEPUASAN" in chunk:
        section = "Kepuasan Kinerja"
    elif "EFISIENSI" in chunk:
        section = "Kebijakan Efisiensi"
    elif "BANSOS" in chunk or "BLT" in chunk:
        section = "Bantuan Sosial"
    elif "KESIMPULAN" in chunk:
        section = "Kesimpulan"
    return {"source": DOC_NAME, "chunk_index": index, "section": section}

# ── TODO: Add all chunks to the collection ────────────────────────────────
ids = [f"chunk_{i}" for i in range(len(my_chunks))]
metadatas = [my_metadata(c, i) for i, c in enumerate(my_chunks)]

my_collection.add(
    ids=ids,
    documents=my_chunks,
    embeddings=my_vectors,
    metadatas=metadatas
)

assert my_collection.count() == len(my_chunks), \
    f"Expected {len(my_chunks)} items in collection, got {my_collection.count()}"

print(f"Indexed {my_collection.count()} chunks into ChromaDB")

Indexed 49 chunks into ChromaDB


/tmp/ipykernel_6171/2557028979.py:16: DeprecationWarning: The class MyEmbeddingFn does not implement __init__. This will be required in a future version.
  embedding_function=MyEmbeddingFn(),


In [33]:
# ── Step 2c: Verify search works ─────────────────────────────────────────

def my_dense_search(query: str, k: int = 5, where: dict = None) -> list[dict]:

    # ── TODO: implement this function ─────────────────────────────────────
    query_vector = genai.embed_content(
        model="models/gemini-embedding-2",
        content=query,
        task_type="retrieval_query"
    )['embedding']

    results = my_collection.query(
        query_embeddings=[query_vector],
        n_results=k,
        where=where
    )

    hits = []
    if results['documents'] and results['documents'][0]:
        for i in range(len(results['documents'][0])):
            hits.append({
                "text": results['documents'][0][i],
                "score": results['distances'][0][i] if 'distances' in results and results['distances'] else 0.0,
                "metadata": results['metadatas'][0][i] if 'metadatas' in results and results['metadatas'] else {},
                "chunk_index": results['metadatas'][0][i].get('chunk_index', i)
            })
    return hits


# ── TODO: Run a test search using a topic from your document ─────────────
TEST_QUERY = "Berapa persen tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto?"

assert TEST_QUERY, "Set TEST_QUERY to a question about your document"

test_results = my_dense_search(TEST_QUERY, k=3)

assert test_results and len(test_results) > 0, "my_dense_search() returned no results"

print(f"Test query: '{TEST_QUERY}'")
print()
for i, r in enumerate(test_results):
    print(f"  Rank #{i+1} | score={r['score']:.4f}")
    print(f"  {r['text'][:140]}...")
    print()

Test query: 'Berapa persen tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto?'

  Rank #1 | score=0.1548
  KEPUASAN ATAS KINERJA PRESIDEN PRABOWO SUBIANTO
Secara umum, tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto dalam satu t...

  Rank #2 | score=0.1997
  ALASAN UTAMA KEPUASAN KINERJA PRESIDEN
Di antara responden yang merasa puas dengan kinerja Presiden Prabowo Subianto, alasan utama yang pali...

  Rank #3 | score=0.2120
  
LAPORAN RILIS SURVEI NASIONAL INDIKATOR POLITIK INDONESIA
EVALUASI PUBLIK ATAS KINERJA SATU TAHUN PEMERINTAHAN PRABOWO-GIBRAN
Periode Surve...



---
## Step 3 — Hybrid Search (BM25 + Dense + RRF)

**Your task:**
- Build a BM25 index over `my_chunks`
- Implement `my_bm25_search()` using `rank_bm25`
- Implement `my_hybrid_search()` that fuses BM25 + dense results using the `rrf_fusion()` function from the demo
- Run a comparison: show how BM25, Dense, and Hybrid differ on the same query

**Recall:** `rrf_fusion()` is already defined in the demo section above.

In [34]:
# ── Step 3a: Build BM25 index ─────────────────────────────────────────────
from rank_bm25 import BM25Okapi
import re

def simple_tokenize(text: str) -> list[str]:
    return re.sub(r'[^\w\s]', ' ', text.lower()).split()

# ── TODO: Build the BM25 index over my_chunks ────────────────────────────
my_bm25_index = BM25Okapi([simple_tokenize(c) for c in my_chunks])

assert my_bm25_index is not None, "Build the BM25 index"
print("BM25 index built")

BM25 index built


In [35]:
# ── Step 3b: Implement BM25 search ───────────────────────────────────────

def my_bm25_search(query: str, k: int = 5) -> list[dict]:

    # ── TODO: implement this function ─────────────────────────────────────
    tokens = simple_tokenize(query)
    scores = my_bm25_index.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    hits = []
    for idx in top_indices:
        hits.append({
            "text": my_chunks[idx],
            "score": float(scores[idx]),
            "metadata": my_collection.get(ids=[f"chunk_{idx}"])['metadatas'][0],
            "chunk_index": idx
        })
    return hits


# Definition of rrf_fusion to ensure notebook runs standalone
def rrf_fusion(results_lists: list[list[dict]], k: int = 60) -> list[dict]:
    rrf_scores = {}
    doc_map = {}
    for results in results_lists:
        for rank, hit in enumerate(results):
            doc_id = hit['chunk_index']
            doc_map[doc_id] = hit
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0.0
            rrf_scores[doc_id] += 1.0 / (k + rank + 1)
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    fused = []
    for doc_id, score in sorted_docs:
        hit = doc_map[doc_id].copy()
        hit['rrf_score'] = score
        fused.append(hit)
    return fused


# ── Step 3c: Implement hybrid search ─────────────────────────────────────

def my_hybrid_search(query: str, k: int = 5, dense_k: int = 10, bm25_k: int = 10) -> list[dict]:

    # ── TODO: implement this function ─────────────────────────────────────
    dense_hits = my_dense_search(query, k=dense_k)
    bm25_hits = my_bm25_search(query, k=bm25_k)
    fused = rrf_fusion([dense_hits, bm25_hits])
    return fused[:k]


# ── TODO: Test with a query where exact keywords matter ──────────────────
KEYWORD_QUERY = "Bantuan Langsung Tunai BLT senilai Rp 300.000"

assert KEYWORD_QUERY, "Set KEYWORD_QUERY"

print(f"Comparison for: '{KEYWORD_QUERY}'")
print()
d_top = my_dense_search(KEYWORD_QUERY, k=1)
b_top = my_bm25_search(KEYWORD_QUERY, k=1)
h_top = my_hybrid_search(KEYWORD_QUERY, k=1)

assert d_top and b_top and h_top, "All three search functions must return results"

print(f"  Dense  : {d_top[0]['text'][:100]}...")
print(f"  BM25   : {b_top[0]['text'][:100]}...")
print(f"  Hybrid : {h_top[0]['text'][:100]}...")

Comparison for: 'Bantuan Langsung Tunai BLT senilai Rp 300.000'

  Dense  : PENERIMAAN BANTUAN SOSIAL (BANSOS) SELAMA TAHUN 2025...
  BM25   : BLIK TERHADAP PROGRAM BANTUAN LANGSUNG TUNAI (BLT)
Kebijakan pemberian tambahan Bantuan Langsung Tun...
  Hybrid : BLIK TERHADAP PROGRAM BANTUAN LANGSUNG TUNAI (BLT)
Kebijakan pemberian tambahan Bantuan Langsung Tun...


---
## Step 4 — Build the Grounded RAG Chain

**Your task:**
- Write a system prompt appropriate for your document topic
- Implement a `my_rag()` function that uses `my_hybrid_search()` for retrieval
- Answer **5 meaningful questions** about your document
- Test the refusal: ask one question whose answer is NOT in the document

**Good questions to ask:**
- A specific factual question (a number, date, or name from the document)
- A summary question ("What are the main points of section X?")
- A comparative question ("How does X compare to Y?")
- A causal question ("Why did X happen?")
- An implication question ("What does X mean for the future?")

In [36]:
# ── Step 4a: Write your system prompt ─────────────────────────────────────

# ── TODO: Write a domain-appropriate system prompt for your document ──────
MY_SYSTEM_PROMPT = """
You are an Indonesian survey data analyst representative of Indikator Politik Indonesia.
Answer the question strictly using the provided [CONTEXT] facts, percentages, and statistics.
If the required statistics or answers are not directly present in the context, reply exactly with: "Saya tidak memiliki informasi yang cukup."
Do not make up figures or utilize external knowledge.
Cite the chunk section at the end of every claim (e.g., [Evaluasi Kondisi Nasional] or [Kepuasan Kinerja]).
"""

assert len(MY_SYSTEM_PROMPT.strip()) > 50, \
    "MY_SYSTEM_PROMPT is too short. Write a real grounding prompt."

print("System prompt set")
print(MY_SYSTEM_PROMPT)

System prompt set

You are an Indonesian survey data analyst representative of Indikator Politik Indonesia.
Answer the question strictly using the provided [CONTEXT] facts, percentages, and statistics.
If the required statistics or answers are not directly present in the context, reply exactly with: "Saya tidak memiliki informasi yang cukup."
Do not make up figures or utilize external knowledge.
Cite the chunk section at the end of every claim (e.g., [Evaluasi Kondisi Nasional] or [Kepuasan Kinerja]).



In [37]:
# Definition of standalone chat function using Google Gemini API
def chat(messages: list[dict]) -> str:
    system_instruction = ""
    contents = []
    for m in messages:
        if m['role'] == 'system':
            system_instruction = m['content']
        elif m['role'] == 'user':
            contents.append(m['content'])
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_instruction
    )
    response = model.generate_content(contents)
    return response.text


# ── Step 4b: Implement my_rag() ───────────────────────────────────────────

def my_format_context(hits: list[dict]) -> str:
    """Format retrieved chunks into a readable context block."""
    # ── TODO: implement this helper ───────────────────────────────────────
    blocks = []
    for idx, hit in enumerate(hits):
        sect = hit['metadata'].get('section', 'Umum')
        blocks.append(f"[DOCUMENT CHUNK {idx+1}] (Section: {sect})\n{hit['text']}")
    return "\n\n---\n\n".join(blocks)


def my_rag(question: str, k: int = 5) -> dict:
    """
    Full RAG pipeline for your document.
    Returns: {question, answer, hits}
    """
    # ── TODO: implement this function ─────────────────────────────────────
    hits = my_hybrid_search(question, k=k)
    context = my_format_context(hits)
    messages = [
        {"role": "system", "content": MY_SYSTEM_PROMPT},
        {"role": "user", "content": f"[CONTEXT]\n{context}\n\n[PERTANYAAN]\n{question}"}
    ]
    answer = chat(messages)
    return {"question": question, "answer": answer, "hits": hits}


print("my_rag() defined")

my_rag() defined


In [38]:
# ── Step 4c: Answer 5 questions about your document ───────────────────────
import time

# ── TODO: Write 5 meaningful questions about your document ────────────────
MY_QUESTIONS = [
    "Berapa persen tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto?",
    "Apa saja empat masalah paling mendesak yang harus diselesaikan menurut mayoritas warga?",
    "Bagaimana perbandingan tingkat kepuasan kinerja presiden di wilayah Kalimantan dibandingkan DKI Jakarta?",
    "Mengapa program Makan Bergizi Gratis MBG sempat mendapatkan kritik atau penolakan menurut kesimpulan?",
    "Apa dampak pemberian bantuan sosial bansos terhadap persepsi publik atas kondisi ekonomi nasional menurut data kesimpulan?"
]

assert all(q.strip() for q in MY_QUESTIONS), \
    "All 5 questions must be filled in (no empty strings)"

# Run all questions with a 12-second delay to avoid rate-limiting (429)
for idx, q in enumerate(MY_QUESTIONS):
    # Jeda 12 detik
    if idx > 0:
        print("Menunggu jeda 12 detik agar tidak terkena limit kuota (429)...")
        time.sleep(12)

    try:
        result = my_rag(q)
        print(f"{'='*68}")
        print(f"Q: {q}")
        print(f"{'─'*68}")
        print(f"A: {result['answer']}")
        print(f"   [Retrieved {len(result['hits'])} chunks | top sim: {result['hits'][0].get('rrf_score', '?')}]")
        print()
    except Exception as e:
        print(f"Gagal memproses pertanyaan: '{q}' karena error: {e}")
        print("Mencoba menjeda lebih lama (20 detik).")
        time.sleep(20)

Q: Berapa persen tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto?
────────────────────────────────────────────────────────────────────
A: Tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto adalah 77.7% (Sangat Puas 17.3%, Cukup Puas 60.4%). [Kepuasan Kinerja]
   [Retrieved 5 chunks | top sim: 0.03278688524590164]

Menunggu jeda 12 detik agar tidak terkena limit kuota (429)...
Q: Apa saja empat masalah paling mendesak yang harus diselesaikan menurut mayoritas warga?
────────────────────────────────────────────────────────────────────
A: Menurut responden, empat masalah paling mendesak yang harus diselesaikan oleh pemimpin nasional lima tahun ke depan adalah:
1.  Pemberantasan korupsi sebesar 25.5% [Masalah Mendesak]
2.  Mengendalikan harga-harga kebutuhan pokok sebesar 22.3% [Masalah Mendesak]
3.  Menyediakan lapangan kerja atau mengatasi pengangguran sebesar 15.6% [Masalah Mendesak]
4.  Mengurangi kemiskinan sebesar 13.4% [Masalah Mendesak]
   [Retrie

In [39]:
# ── Step 4d: Test out-of-scope refusal ────────────────────────────────────

# ── TODO: Ask something that is clearly NOT in your document ─────────────
OUT_OF_SCOPE_Q = "Berapa persen tingkat inflasi tahunan di negara Malaysia pada tahun 2024?"

assert OUT_OF_SCOPE_Q.strip(), "Set OUT_OF_SCOPE_Q"

result = my_rag(OUT_OF_SCOPE_Q)
print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print()
print("Did the model correctly refuse instead of hallucinating? (check above)")

Q: Berapa persen tingkat inflasi tahunan di negara Malaysia pada tahun 2024?
A: Saya tidak memiliki informasi yang cukup.

Did the model correctly refuse instead of hallucinating? (check above)


---
## Step 5 — Structured Citation Output

**Your task:**
- Implement `my_cited_rag()` that returns a **JSON object** with `answer`, `citations`, and `has_sufficient_context` fields
- Run it on 2 of your 5 questions
- Parse and display the citations cleanly

**Expected JSON shape:**
```json
{
  "answer": "Revenue grew 23% ...",
  "citations": [
    {"claim": "Revenue grew 23%", "source": "my_doc.pdf", "section": "executive_summary"}
  ],
  "has_sufficient_context": true
}
```

In [41]:
# ── Step 5: Cited RAG with structured JSON output ─────────────────────────
import json

# ── TODO: Write a citation system prompt ─────────────────────────────────
MY_CITATION_SYSTEM = """
You are a structured data survey analysis assistant.
You must return ONLY a JSON object (no markdown formatting, no prose prefix/suffix).

Define the exact JSON schema:
{
  "answer": "string containing the detailed grounded Indonesian answer based strictly on context",
  "citations": [
    {
      "claim": "string representing the fact or statistics claimed in your answer",
      "source": "string representing source filename from context metadata",
      "section": "string representing the section label from context metadata"
    }
  ],
  "has_sufficient_context": true or false
}
"""

def my_cited_rag(question: str, k: int = 5) -> dict:
    ls back gracefully if JSON parsing fails.
    """
    # ── TODO: implement ───────────────────────────────────────────────────
    hits = my_hybrid_search(question, k=k)
    context = my_format_context(hits)

    # Gemini Native JSON mode ensures strict JSON formatting is adhered to.
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=MY_CITATION_SYSTEM
    )
    response = model.generate_content(
        f"[CONTEXT]\n{context}\n\n[PERTANYAAN]\n{question}",
        generation_config={"response_mime_type": "application/json"}
    )

    try:
        parsed = json.loads(response.text.strip())
        return parsed
    except Exception as e:
        # Manual parsing fence fallback if schema output fails
        raw = response.text.strip()
        if raw.startswith("```json"):
            raw = raw[7:]
        if raw.endswith("```"):
            raw = raw[:-3]
        try:
            return json.loads(raw.strip())
        except:
            return {"answer": raw, "citations": [], "has_sufficient_context": False}


# ── Run on your first 2 questions ────────────────────────────────────────
for q in MY_QUESTIONS[:2]:
    result = my_cited_rag(q)
    print(f"Q: {q}")
    print(f"A: {result.get('answer', 'N/A')}")
    print(f"Sufficient context: {result.get('has_sufficient_context')}")
    for c in result.get('citations', []):
        print(f"  • '{c.get('claim','')[:60]}' -> {c.get('source','')} [{c.get('section','')}]")
    print()

Q: Berapa persen tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto?
A: Tingkat kepuasan publik terhadap kinerja Presiden Prabowo Subianto adalah sekitar 77.7%, di mana 17.3% merasa sangat puas dan 60.4% merasa cukup puas.
Sufficient context: True
  • 'Sekitar 77.7% warga merasa puas (Sangat Puas 17.3%, Cukup Pu' -> DOCUMENT CHUNK 1 [Kepuasan Kinerja]

Q: Apa saja empat masalah paling mendesak yang harus diselesaikan menurut mayoritas warga?
A: Menurut responden, empat masalah paling mendesak yang harus diselesaikan oleh pemimpin nasional lima tahun ke depan adalah pemberantasan korupsi sebesar 25.5%, mengendalikan harga-harga kebutuhan pokok sebesar 22.3%, menyediakan lapangan kerja atau mengatasi pengangguran sebesar 15.6%, dan mengurangi kemiskinan sebesar 13.4%.
Sufficient context: True
  • 'Pemberantasan korupsi sebesar 25.5%' -> DOCUMENT CHUNK 2 [Masalah Mendesak]
  • 'Mengendalikan harga-harga kebutuhan pokok sebesar 22.3%' -> DOCUMENT CHUNK 2 [Masalah Mendesak]


---
## Step 6 — Demonstrate 2 Failure Modes (and Their Fixes)

**Your task:** Choose any **2** of the 4 failure modes below. For each one:
1. Show the **broken** version (incorrect or missing output)
2. Show the **fixed** version (correct output)
3. Write a 1–2 sentence explanation of *why* the fix works

| Failure mode | How to trigger it |
|---|---|
| Hallucination | Use a weak system prompt + ask out-of-scope question |
| Context overflow | Set k to the total number of chunks |
| Format error | Ask for JSON without providing a schema |
| Retrieval failure | Ask with an acronym or synonym that BM25 cannot match |

In [43]:
# ── Step 6a: Failure Mode 1 ───────────────────────────────────────────────
# ── TODO: Choose one failure mode and demonstrate it ─────────────────────

FAILURE_1_NAME = "Hallucination on Out-of-Scope Query"

print(f"FAILURE MODE 1: {FAILURE_1_NAME}")
print("="*60)
print()

# Weak System prompt that does not restrict parametric knowledge
WEAK_PROMPT = "Jawab pertanyaan user sebaik mungkin."
def weak_rag(question: str) -> str:
    model = genai.GenerativeModel(model_name="gemini-2.5-flash", system_instruction=WEAK_PROMPT)
    response = model.generate_content(question)
    return response.text

print("❌ Broken:")
print(weak_rag("Siapa pemenang Pemilu Presiden Amerika Serikat pada tahun 2024?"))

print()
print("✅ Fixed:")
print(my_rag("Siapa pemenang Pemilu Presiden Amerika Serikat pada tahun 2024?")["answer"])

print()
# ── TODO: Explain why the fix works ──────────────────────────────────────
EXPLANATION_1 = "The weak system prompt allows the LLM to pull knowledge from its own pre-trained weights. Enforcing strict grounding instructions in MY_SYSTEM_PROMPT forces the model to refuse to answer queries not backed up by Context."
print(f"Why the fix works: {EXPLANATION_1}")

FAILURE MODE 1: Hallucination on Out-of-Scope Query

❌ Broken:
Pemilu Presiden Amerika Serikat tahun 2024 **belum berlangsung**.

Pemilihan umum tersebut dijadwalkan akan diadakan pada hari **Selasa, 5 November 2024**.

Oleh karena itu, belum ada pemenang yang bisa diumumkan saat ini. Pemenang baru akan diketahui setelah pemilihan selesai, suara dihitung, dan hasilnya diverifikasi.

✅ Fixed:
Saya tidak memiliki informasi yang cukup.

Why the fix works: The weak system prompt allows the LLM to pull knowledge from its own pre-trained weights. Enforcing strict grounding instructions in MY_SYSTEM_PROMPT forces the model to refuse to answer queries not backed up by Context.


In [45]:
# ── Step 6b: Failure Mode 2 ───────────────────────────────────────────────
# ── TODO: Choose a DIFFERENT failure mode ────────────────────────────────

FAILURE_2_NAME = "Format Error on Structured JSON Output"

assert FAILURE_2_NAME != FAILURE_1_NAME, "Choose a different failure mode for Step 6b"

print(f"FAILURE MODE 2: {FAILURE_2_NAME}")
print("="*60)
print()

# Weak instruction asking for JSON without structured output config
def broken_json_rag(question: str) -> str:
    model = genai.GenerativeModel(model_name="gemini-2.5-flash", system_instruction="Berikan jawaban dalam format JSON.")
    response = model.generate_content(question)
    return response.text

print("❌ Broken:")
print(broken_json_rag("Sebutkan empat masalah paling mendesak bagi bangsa."))

print()
print("✅ Fixed:")
import pprint
pprint.pprint(my_cited_rag("Sebutkan empat masalah paling mendesak bagi bangsa."))

print()
EXPLANATION_2 = "Requesting JSON output through standard generation prompts usually produces markdown-wrapped text blocks. Configuring response_mime_type to application/json in Gemini strictly forces the engine to output parsesable, clean JSON."
print(f"Why the fix works: {EXPLANATION_2}")

FAILURE MODE 2: Format Error on Structured JSON Output

❌ Broken:
```json
{
  "masalahMendesakNasional": [
    {
      "nama": "Ketimpangan Ekonomi dan Kemiskinan",
      "deskripsi": "Kesenjangan pendapatan yang lebar antara kelompok masyarakat kaya dan miskin, tingginya angka kemiskinan, serta kurangnya kesempatan kerja yang layak yang menghambat pertumbuhan inklusif dan stabilitas sosial."
    },
    {
      "nama": "Korupsi dan Tata Kelola Pemerintahan yang Lemah",
      "deskripsi": "Praktik korupsi yang merajalela di berbagai tingkatan, birokrasi yang tidak efisien, kurangnya transparansi, dan penegakan hukum yang lemah, yang menghambat pembangunan, investasi, dan kepercayaan publik."
    },
    {
      "nama": "Akses dan Kualitas Pendidikan serta Layanan Kesehatan",
      "deskripsi": "Keterbatasan akses terhadap pendidikan berkualitas tinggi dan layanan kesehatan yang memadai, terutama di daerah terpencil atau bagi kelompok rentan, serta kualitas fasilitas dan tenaga yang belum

---
## Step 7 — Reflection

Answer the 3 short-answer questions below. Write directly in the markdown cell (double-click to edit).

### ✍️ Reflection Questions

**Q1: What chunk size did you choose and why was it appropriate for your document type?**

> Saya memilih CHUNK_SIZE = 500 dan CHUNK_OVERLAP = 50. Ukuran chunk ini optimal karena dokumen rilis survei disusun per topik slide yang padat dengan data persentase statistik. Ukuran 500 karakter memastikan satu slide hasil survei tetap utuh berada dalam satu potongan chunk tanpa terputus secara sembarangan, sementara overlap 50 karakter mencegah hilangnya konteks di batas peralihan topik.


---

**Q2: Did hybrid search return noticeably different results than pure dense search on any of your queries? Describe one example.**

> Ya. Pada query kata kunci eksak seperti 'BLT Rp 300.000', BM25 berhasil secara langsung mencocokkan kemunculan istilah angka 'Rp 300.000' dan menempatkan chunk slide BLT di posisi teratas. Di sisi lain, dense search murni terkadang mengembalikan dokumen bantuan sosial lainnya (seperti PKH atau bansos beras) karena kesamaan makna semantik 'bantuan'. Hybrid search dengan RRF berhasil mempertahankan akurasi kata kunci eksak sekaligus konteks semantiknya.


---

**Q3: If you were deploying this RAG system to real users, what one improvement would you prioritise first and why?**

> Saya akan memprioritaskan integrasi tahap Re-ranking (seperti Cohere Rerank atau Cross-Encoder). Meskipun pencarian hybrid menggunakan RRF telah menghasilkan peringkat awal yang mumpuni, model re-ranking akan mengevaluasi interaksi kueri-dokumen secara mendalam pada 10-15 kandidat teratas sebelum dikirim ke model Gemini, sehingga meningkatkan kualitas relevansi dan meminimalisir distraksi.

In [46]:
# ── Final submission check ────────────────────────────────────────────────
# Run this cell last. All assertions must pass before submitting.

checks = []

# Step 0
checks.append(("Document loaded",          len(MY_DOCUMENT.strip()) >= 500))
checks.append(("DOC_NAME set",             bool(DOC_NAME.strip())))

# Step 1
checks.append(("Chunks created",           my_chunks is not None and len(my_chunks) >= 3))
checks.append(("Chunk size set",           CHUNK_SIZE > 0 and CHUNK_OVERLAP > 0))

# Step 2
checks.append(("Vectors created",          my_vectors is not None and len(my_vectors) == len(my_chunks)))
checks.append(("ChromaDB indexed",         my_collection.count() == len(my_chunks)))
checks.append(("dense_search working",     bool(my_dense_search(MY_QUESTIONS[0], k=1))))

# Step 3
checks.append(("BM25 index built",         my_bm25_index is not None))
checks.append(("bm25_search working",      bool(my_bm25_search(MY_QUESTIONS[0], k=1))))
checks.append(("hybrid_search working",    bool(my_hybrid_search(MY_QUESTIONS[0], k=1))))

# Step 4
checks.append(("System prompt written",    len(MY_SYSTEM_PROMPT.strip()) > 50))
checks.append(("5 questions filled",       all(q.strip() for q in MY_QUESTIONS)))
checks.append(("my_rag() works",           bool(my_rag(MY_QUESTIONS[0]).get('answer'))))

# Step 5
checks.append(("Citation system prompt",   len(MY_CITATION_SYSTEM.strip()) > 50))

# Step 6
checks.append(("Failure mode 1 named",     bool(FAILURE_1_NAME.strip())))
checks.append(("Failure mode 2 named",     bool(FAILURE_2_NAME.strip())))
checks.append(("Different failure modes",  FAILURE_1_NAME != FAILURE_2_NAME))

# Results
print("\n📋 Submission Checklist")
print("=" * 50)
all_pass = True
for name, result in checks:
    icon = "✅" if result else "❌"
    print(f"  {icon}  {name}")
    if not result:
        all_pass = False

print()
if all_pass:
    print("All checks passed! Your notebook is ready to submit.")
    print(f"   Save as: RAG_Assignment_{DOC_TOPIC.replace(' ', '_')[:30]}.ipynb")
else:
    print("Some checks failed. Fix the items above before submitting.")


📋 Submission Checklist
  ✅  Document loaded
  ✅  DOC_NAME set
  ✅  Chunks created
  ✅  Chunk size set
  ✅  Vectors created
  ✅  ChromaDB indexed
  ✅  dense_search working
  ✅  BM25 index built
  ✅  bm25_search working
  ✅  hybrid_search working
  ✅  System prompt written
  ✅  5 questions filled
  ✅  my_rag() works
  ✅  Citation system prompt
  ✅  Failure mode 1 named
  ✅  Failure mode 2 named
  ✅  Different failure modes

All checks passed! Your notebook is ready to submit.
   Save as: RAG_Assignment_Survei_Nasional_Evaluasi_Satu_.ipynb
